In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


# Verify the Jacobian Approximation of Measurement Equation

In [ ]:
from pypower.api import case14, case57, case118, ext2int, makeYbus, runopf
from pypower.idx_brch import *
from pypower.idx_bus import *
import scipy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Choice of test case
# case = case14()
case = case57()
# case = case118()
# No transformer
case['branch'][:,TAP] = 0
case['branch'][:,SHIFT] = 0
#case['branch'][:,BR_B] = 0
case['bus'][:,GS] = 0
case['bus'][:,BS] = 0
case_int = ext2int(case)
ys = 1/(case['branch'][:,BR_R]+1j*case['branch'][:,BR_X])
gs = np.real(ys)
bs = np.imag(ys)
gsh = case['bus'][:,GS]/case['baseMVA']
bsh = case['bus'][:,BS]/case['baseMVA']
bc = case_int['branch'][:,BR_B]

Ybus, Yf, Yt = makeYbus(case_int['baseMVA'], case_int['bus'], case_int['branch'])
# To numpy
Ybus = scipy.sparse.csr_matrix.todense(Ybus).getA()
Yf = scipy.sparse.csr_matrix.todense(Yf).getA()
Yt = scipy.sparse.csr_matrix.todense(Yt).getA()
# Branch Incidence Matrix
f_bus = case_int['branch'][:, 0].astype('int')        # list of "from" buses
t_bus = case_int['branch'][:, 1].astype('int')        # list of "to" buses

no_bus = len(case['bus'])
no_brh = len(case['branch'])

Cf = np.zeros((no_brh,no_bus))         # "from" bus incidence matrix
Ct = np.zeros((no_brh,no_bus))         # "to" bus incidence matrix
for i in range(no_brh):
    Cf[i,f_bus[i]] = 1
    Ct[i,t_bus[i]] = 1
A = Cf-Ct

In [ ]:
result = runopf(case)
pf_result = result['branch'][:,PF]/100
print(pf_result)

In [ ]:
# Active power
# From complex power
vmag = result['bus'][:,VM]
vang = result['bus'][:,VA]*np.pi/180  # In radian
vcomplex = vmag*np.exp(1j*vang)
Sf = np.diag(Cf@vcomplex)@np.conjugate(Yf)@np.conjugate(vcomplex)
Yf1 = np.diag(ys)@A+1j*np.diag(bc/2)@Cf
Yf2 = np.diag(ys)@A
Sf2 = np.diag(Cf@vcomplex)@np.conjugate(Yf2)@np.conjugate(vcomplex)

Ybus1 = A.T@np.diag(ys)@A + 1j*Cf.T@np.diag(bc/2)@Cf + 1j*Ct.T@np.diag(bc/2)@Ct
Ybus2 = A.T@np.diag(ys)@A

In [ ]:
Ybus1 - Ybus2

In [ ]:
np.imag(Sf) - np.imag(Sf2)

In [ ]:
15.26308652 + 4.23498368

In [ ]:
np.imag(Ybus)

In [ ]:
# Active power
# From complex power
vmag = result['bus'][:,VM]
vang = result['bus'][:,VA]*np.pi/180  # In radian
vcomplex = vmag*np.exp(1j*vang)
Sf = np.diag(Cf@vcomplex)@np.conjugate(Yf)@np.conjugate(vcomplex)

# From active power
Pf = -(np.diag(Cf@vmag)@np.diag(Ct@vmag)@(np.diag(gs)@np.cos(A@vang) + np.diag(bs-bc/2)@np.sin(A@vang)) - gs*(Cf@vmag**2))
Qf = -(np.diag(Cf@vmag)@np.diag(Ct@vmag)@(np.diag(gs)@np.sin(A@vang) - np.diag(bs-bc/2)@np.cos(A@vang)) + bs*(Cf@vmag**2))
plt.plot(Qf)
plt.plot(np.imag(Sf))

In [ ]:
np.imag(Sf) - Qf

In [ ]:
Pf - pf_result

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.real(Sf), label = 'complex')
plt.plot(Pf, label = 'direct')
plt.legend()

In [ ]:
Pf - np.real(Sf)

In [ ]:
bs

In [ ]:
np.imag(Yf)